In [ ]:
from pathlib import Path
import warnings

import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb

warnings.filterwarnings("ignore")

print("Libraries loaded.")

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "chfdb" / "files"
FEATURES_DIR = PROJECT_ROOT / "features"

FEATURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)
print("Features folder:", FEATURES_DIR)
print("Dataset exists:", DATA_DIR.exists())

Project root: C:\Users\othma\HF-Watch-AI
Data folder: C:\Users\othma\HF-Watch-AI\data\chfdb\files
Features folder: C:\Users\othma\HF-Watch-AI\features
Dataset exists: True


In [3]:
patient_id = "chf01"

record = wfdb.rdrecord(str(DATA_DIR / patient_id))

sampling_rate = int(record.fs)
window_seconds = 300
window_samples = sampling_rate * window_seconds

ecg = record.p_signal[:, 0]

number_of_windows = len(ecg) // window_samples

print("Sampling rate:", sampling_rate)
print("Recording samples:", len(ecg))
print("Complete 5-minute windows:", number_of_windows)

Sampling rate: 250
Recording samples: 17994491
Complete 5-minute windows: 239


In [ ]:
all_windows = []
failed_windows = []

WINDOW_SECONDS = 300
MINIMUM_BEATS = 150

for patient_number in range(1, 16):

    patient_id = f"chf{patient_number:02d}"

    print(f"\nProcessing {patient_id}")

    try:

        record = wfdb.rdrecord(str(DATA_DIR / patient_id))

        sampling_rate = int(record.fs)

        ecg = record.p_signal[:, 0]

        window_samples = sampling_rate * WINDOW_SECONDS

        number_of_windows = len(ecg) // window_samples

        print("Windows:", number_of_windows)

        for window_index in range(number_of_windows):

            start = window_index * window_samples
            end = start + window_samples

            segment = ecg[start:end]

            if np.isnan(segment).mean() > 0.05:
                continue

            segment = (
                pd.Series(segment)
                .interpolate(limit_direction="both")
                .to_numpy()
            )

            cleaned = nk.ecg_clean(
                segment,
                sampling_rate=sampling_rate
            )

            _, info = nk.ecg_process(
                cleaned,
                sampling_rate=sampling_rate
            )

            beats = len(info["ECG_R_Peaks"])

            if beats < MINIMUM_BEATS:
                continue

            hrv = nk.hrv(
                info,
                sampling_rate=sampling_rate,
                show=False
            )

            hrv.insert(0, "patient_id", patient_id)
            hrv.insert(1, "window", window_index)
            hrv.insert(2, "beats", beats)
            hrv.insert(3, "label", 1)

            all_windows.append(hrv)

    except Exception as e:

        failed_windows.append(
            {
                "patient": patient_id,
                "error": str(e)
            }
        )


Processing chf01
Windows: 239

Processing chf02
Windows: 237
